# 🐍 Day 02 — Solutions & Explanations

Companion to **day02-python-interview-drills-and-gotchas.ipynb**.

**How to use this:**
1. Try the problem yourself first (even 10 minutes of struggling helps you remember it).
2. Stuck? Read the **idea** first, not the code. Often the idea alone unlocks it.
3. Then read the solution, then the **step-by-step walkthrough**.
4. Close this notebook and re-write the solution from scratch. If you can, you actually learned it.

Every solution cell is runnable and has the original tests, so you can run it and see the ✅.

> ⚠️ **Heads-up:** The original notebook's explanation for Gotcha 2.3 says `4` is *not* removed. That's wrong for that exact list. See my note in 2.3 below.

---
# 🛠️ Section 1 — Core Data Structures

## 1.1 Sorting with `lambda`

**Idea:** `sorted()` needs to know *what to compare*. `key=` tells it. A `lambda` is just a tiny throwaway function that takes one item and returns the thing to compare.

For each employee dict, we return `emp["salary"]`, so Python sorts by salary.

In [ ]:
from typing import List, Dict, Any

def sort_employees_by_salary(employees: List[Dict[str, Any]], descending: bool = True) -> List[Dict[str, Any]]:
    return sorted(employees, key=lambda emp: emp["salary"], reverse=descending)

# ---- tests (same as original) ----
sample_employees = [
    {"name": "Pavan", "dept": "Data", "salary": 65000},
    {"name": "Amit", "dept": "QA", "salary": 45000},
    {"name": "Sneha", "dept": "Data", "salary": 85000},
    {"name": "Rahul", "dept": "DevOps", "salary": 55000}
]

res = sort_employees_by_salary(sample_employees, descending=True)
assert res[0]["name"] == "Sneha", "Highest earner should be first"
assert res[-1]["name"] == "Amit", "Lowest earner should be last"
assert res[1]["salary"] == 65000

print("✅ 1.1 Sorting with lambda passed!")

### 🔍 Walkthrough
- `lambda emp: emp["salary"]` is the same as writing:
  ```python
  def get_salary(emp):
      return emp["salary"]
  ```
- `sorted()` calls it on each dict → gets `65000, 45000, 85000, 55000` → sorts by those numbers.
- `reverse=True` flips it to biggest-first. We pass `descending` straight in as `reverse`.

### 🧠 Remember
| | Returns | Changes original? |
|---|---|---|
| `sorted(x, ...)` | new list | No |
| `x.sort(...)` | `None` | Yes (in-place) |

**Common mistake:** `employees = employees.sort(...)` → sets `employees` to `None`. 😅

## 1.2 Safe dictionary access with `.get()`

**Idea:** `d["status"]` crashes with `KeyError` if the key is missing. `d.get("status")` just returns `None` instead. Then we swap `None` for `"UNKNOWN"`.

In [ ]:
def extract_user_status(user_record: dict) -> str:
    status = user_record.get("status")
    return status if status else "UNKNOWN"

# ---- tests ----
assert extract_user_status({"id": 1, "status": "ACTIVE"}) == "ACTIVE"
assert extract_user_status({"id": 2}) == "UNKNOWN", "Missing key should return UNKNOWN"
assert extract_user_status({"id": 3, "status": None}) == "UNKNOWN", "None value should fallback to UNKNOWN"

print("✅ 1.2 Safe dictionary lookups passed!")

### 🔍 Walkthrough
| Input | `.get("status")` gives | `if status` is | Returns |
|---|---|---|---|
| `{"status": "ACTIVE"}` | `"ACTIVE"` | truthy | `"ACTIVE"` |
| `{"id": 2}` (no key) | `None` | falsy | `"UNKNOWN"` |
| `{"status": None}` | `None` | falsy | `"UNKNOWN"` |

### 🧠 Why not just `.get("status", "UNKNOWN")`?
That one-liner handles the **missing key** case but **not** `{"status": None}`, because the key *exists*, so the default is ignored and you get `None` back. That's exactly what test 3 checks. A classic trap.

Side effect of our version: an empty string `""` is also falsy, so it becomes `"UNKNOWN"` too. The task says "not empty", so that's what we want.

## 1.3 Set reconciliation (source vs target)

**Idea:** Turn both lists into sets, then use set math:
- `A - B` → in A but not in B
- `A & B` → in both

In [ ]:
from typing import List, Dict, Any

def reconcile_ids(source_ids: List[int], target_ids: List[int]) -> Dict[str, Any]:
    src_set = set(source_ids)
    tgt_set = set(target_ids)
    return {
        "missing_in_target": src_set - tgt_set,
        "new_in_target": tgt_set - src_set,
        "matching_count": len(src_set & tgt_set),
    }

# ---- tests ----
src = [101, 102, 103, 104, 105]
tgt = [103, 104, 105, 106, 107]

report = reconcile_ids(src, tgt)
assert report["missing_in_target"] == {101, 102}
assert report["new_in_target"] == {106, 107}
assert report["matching_count"] == 3  # 103, 104, 105

print("✅ 1.3 Set reconciliation passed!")

### 🔍 Walkthrough
```
source = {101, 102, 103, 104, 105}
target = {103, 104, 105, 106, 107}

source - target = {101, 102}          ← source has them, target doesn't  (missing_in_target)
target - source = {106, 107}          ← target has them, source doesn't   (new_in_target)
source & target = {103, 104, 105}     ← both  → len = 3                   (matching_count)
```

### 🧠 Why sets and not lists?
- `x in some_list` checks items one by one → slow on millions of rows.
- `x in some_set` is a hash lookup → basically instant.
- Bonus: converting to a set also removes duplicates for free.

**SQL mapping (good to say in interviews):** `A - B` ≈ `EXCEPT` or `LEFT JOIN ... WHERE b.key IS NULL`. `A & B` ≈ `INTERSECT` / `INNER JOIN`.

## 1.4 Cleaning and parsing dates

**Idea:** Clean the string (`strip`), then *try* each known format. The first one that doesn't blow up wins. If none work, return `None`.

In [ ]:
from datetime import datetime
from typing import Optional

def parse_date_to_standard(date_str: Optional[str]) -> Optional[str]:
    if not date_str or not isinstance(date_str, str):
        return None

    cleaned = date_str.strip()

    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            dt = datetime.strptime(cleaned, fmt)
            return dt.strftime("%Y-%m-%d")
        except ValueError:
            continue   # this format didn't match, try the next one

    return None

# ---- tests ----
assert parse_date_to_standard("  25/12/2023  ") == "2023-12-25"
assert parse_date_to_standard("2023-12-25") == "2023-12-25"
assert parse_date_to_standard("invalid_date") is None
assert parse_date_to_standard(None) is None

print("✅ 1.4 Date parsing passed!")

### 🔍 Walkthrough with `"  25/12/2023  "`
1. Not empty, is a string → continue.
2. `.strip()` → `"25/12/2023"`.
3. Try `"%d/%m/%Y"` → matches → we get a date object for 25 Dec 2023.
4. `.strftime("%Y-%m-%d")` → `"2023-12-25"` ✅

With `"2023-12-25"`:
1. Try `"%d/%m/%Y"` → `ValueError` (no slashes) → `continue`.
2. Try `"%Y-%m-%d"` → matches → return `"2023-12-25"`.

With `"invalid_date"`: both formats fail → falls out of the loop → `return None`.

### 🧠 Remember
- `strptime` = **p**arse (string → datetime)
- `strftime` = **f**ormat (datetime → string)
- Format codes: `%d` day, `%m` month, `%Y` 4-digit year.
- First check `if not date_str` handles `None` and `""` so `.strip()` never gets called on `None` (that would crash).

## 1.5 `zip` headers with rows

**Idea:** `zip(headers, row)` pairs them up position by position. `dict(...)` turns those pairs into a dictionary. Do that for every row.

In [ ]:
from typing import List, Tuple, Dict, Any

def rows_to_dicts(headers: List[str], data_rows: List[Tuple]) -> List[Dict[str, Any]]:
    return [dict(zip(headers, row)) for row in data_rows]

# ---- tests ----
cols = ["id", "name", "role"]
rows = [
    (1, "Pavan", "Data Engineer"),
    (2, "Kalyan", "Analyst")
]
result = rows_to_dicts(cols, rows)
assert result == [
    {"id": 1, "name": "Pavan", "role": "Data Engineer"},
    {"id": 2, "name": "Kalyan", "role": "Analyst"}
]

print("✅ 1.5 rows_to_dicts passed!")

### 🔍 Walkthrough
For the first row:
```python
zip(["id", "name", "role"], (1, "Pavan", "Data Engineer"))
# → ("id", 1), ("name", "Pavan"), ("role", "Data Engineer")

dict(...)
# → {"id": 1, "name": "Pavan", "role": "Data Engineer"}
```
The list comprehension just does this for each row.

Same thing without the one-liner (use this if the comprehension feels confusing):
```python
result = []
for row in data_rows:
    result.append(dict(zip(headers, row)))
return result
```

### 🧠 Remember
`zip` stops at the **shortest** input. If a row has fewer values than headers, the extra headers are silently dropped. Watch for that on real data.

---
# ⚠️ Section 2 — The 6 Gotchas

These have no code to write. Run the cells, see the output, and make sure you can **explain the "why" out loud**. That's what the interviewer is checking.

## 2.1 Aliasing vs shallow copy

In [ ]:
list_a = [1, 2, [3, 4]]
list_b = list_a          # same list, two names
list_c = list_a.copy()   # new outer list, SAME inner list

list_b[0] = 99
list_c[2].append(500)

print("list_a:", list_a)
print("list_b:", list_b)
print("list_c:", list_c)

### ✅ Answer
```
list_a: [99, 2, [3, 4, 500]]
list_b: [99, 2, [3, 4, 500]]
list_c: [1, 2, [3, 4, 500]]
```

### 🔍 Walkthrough
Think of variables as **name tags** stuck on objects, not boxes holding values.

```
list_a ──┐
list_b ──┴──► [1, 2, •]──┐
                          ├──► [3, 4]      ← the inner list (ONE object)
list_c ─────► [1, 2, •]──┘
              (new outer list)
```

1. `list_b = list_a` → no copy at all. Both names stick to the same list.
2. `list_c = list_a.copy()` → a **new outer list** is built, but its third slot still points to the *same* inner `[3, 4]`.
3. `list_b[0] = 99` → changes the shared outer list → `list_a` sees it too. `list_c` doesn't (it has its own outer list).
4. `list_c[2].append(500)` → modifies the **shared inner list** → all three see the `500`.

### 🧠 One-liner for the interview
> "`=` doesn't copy. `.copy()` is shallow, meaning the outer container is new but nested objects are shared. Use `copy.deepcopy()` when you need full independence."

In [ ]:
# Bonus: deepcopy fixes the problem
import copy

x = [1, 2, [3, 4]]
y = copy.deepcopy(x)
y[2].append(500)
print("x:", x)   # unchanged
print("y:", y)

## 2.2 Mutable default argument

In [ ]:
def add_event(event_id, event_list=[]):
    event_list.append(event_id)
    return event_list

print("Call 1:", add_event("login"))
print("Call 2:", add_event("logout"))

### ✅ Answer
```
Call 1: ['login']
Call 2: ['login', 'logout']     ← surprise! login is still there
```

### 🔍 Walkthrough
- Python runs the `def` line **once**, when the function is created. That's when `[]` gets built.
- That one list gets stored on the function itself and reused every time you *don't* pass `event_list`.
- Call 1 appends `"login"` to it. Call 2 appends `"logout"` to the **same** list.

### 🛠️ The fix: use `None` as the default

In [ ]:
def add_event(event_id, event_list=None):
    if event_list is None:
        event_list = []          # fresh list on every call
    event_list.append(event_id)
    return event_list

print("Call 1:", add_event("login"))
print("Call 2:", add_event("logout"))   # now independent

### 🧠 Remember
Never use `[]`, `{}` or `set()` as default arguments. Use `None` and create the object inside the function. (Immutable defaults like `0`, `""`, `None`, `True` are totally fine.)

## 2.3 Modifying a list while looping over it

In [ ]:
numbers = [1, 2, 3, 4, 5, 6]
for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)

print("Remaining numbers:", numbers)

### ✅ Answer
```
Remaining numbers: [1, 3, 5]
```

> ⚠️ **Correction to the original notebook:** It says "`4` was not removed". For *this* list that's not true; the output really is `[1, 3, 5]`. The bug is still real, it just doesn't show with this input. Here's why, then a list where it does show.

### 🔍 Walkthrough (what the loop actually does)
The loop uses a hidden **index** that goes 0, 1, 2, 3... Removing an item shifts everything after it one slot left, but the index still moves forward.

| Step | Index | List before | `n` | Action | List after |
|---|---|---|---|---|---|
| 1 | 0 | `[1,2,3,4,5,6]` | 1 | odd, keep | same |
| 2 | 1 | `[1,2,3,4,5,6]` | 2 | **remove 2** | `[1,3,4,5,6]` |
| 3 | 2 | `[1,3,4,5,6]` | 4 | **remove 4** | `[1,3,5,6]` |
| 4 | 3 | `[1,3,5,6]` | 6 | **remove 6** | `[1,3,5]` |

Notice step 3: the loop jumped straight from index 1 to index 2, so **`3` was skipped**. It just happened to be odd, so no harm done.

### 💥 Where it actually breaks: consecutive evens

In [ ]:
nums = [2, 4, 6, 8]
for n in nums:
    if n % 2 == 0:
        nums.remove(n)

print("Expected: []")
print("Got:     ", nums)   # [4, 8] — the bug shows up!

After removing `2`, the list becomes `[4,6,8]` and the index moves to 1, which is now `6`. So `4` gets skipped. After removing `6` the list is `[4,8]`, index 2 is out of range, and the loop ends. `4` and `8` survive.

### 🛠️ Fixes

In [ ]:
numbers = [1, 2, 3, 4, 5, 6]

# Fix 1 (best): list comprehension — build a new list
only_odds = [x for x in numbers if x % 2 != 0]
print("Fix 1:", only_odds)

# Fix 2: loop over a COPY, modify the original
nums = [2, 4, 6, 8]
for n in nums[:]:
    if n % 2 == 0:
        nums.remove(n)
print("Fix 2:", nums)

### 🧠 Remember
> "Don't change a list while iterating over it. Build a new one (comprehension) or iterate over a copy."

## 2.4 `d['key']` vs `d.get('key')`

In [ ]:
config = {"database": "postgres", "port": 5432}

print("Database:", config["database"])
print("Database (.get):", config.get("database"))

# config["password"]   # <- would crash with KeyError
print("Password with .get():", config.get("password", "default_secret"))

### ✅ Answer
```
Database: postgres
Database (.get): postgres
Password with .get(): default_secret
```

### 🔍 Rule of thumb
| Use | When | Why |
|---|---|---|
| `d["key"]` | The field is **required** (e.g. primary key `id`) | If it's missing, something is really wrong. **Crash loudly** so you notice. |
| `d.get("key", default)` | The field is **optional** (e.g. `middle_name`) | Missing is normal, so use a fallback. |

In a pipeline, silently filling in a missing primary key with a default would hide corrupted data. That's worse than a crash.

## 2.5 Changing a list inside a tuple

In [ ]:
my_tuple = (10, "Pavan", [1, 2])

try:
    my_tuple[2].append(99)
    print("Success! my_tuple is now:", my_tuple)
except TypeError as e:
    print("Error:", e)

### ✅ Answer
```
Success! my_tuple is now: (10, 'Pavan', [1, 2, 99])
```
No error.

### 🔍 Walkthrough
A tuple is immutable in the sense that **which objects it points to can't change**. It stores three "pointers": → `10`, → `"Pavan"`, → the list.

- `my_tuple[2] = [5, 6]` ❌ tries to change what slot 2 points to → `TypeError`.
- `my_tuple[2].append(99)` ✅ doesn't touch the tuple's pointers. It reaches *through* the pointer and changes the list itself.

In [ ]:
my_tuple = (10, "Pavan", [1, 2])
try:
    my_tuple[2] = [5, 6]
except TypeError as e:
    print("Reassigning a slot fails:", e)

### 🧠 Remember
> "Tuples are shallowly immutable. If they hold a mutable object, that object can still change."

Side effect: a tuple containing a list can't be used as a dict key or in a set (it isn't hashable).

## 2.6 `==` vs `is`

In [ ]:
class SneakyObject:
    def __eq__(self, other):
        return True   # claims to equal everything!

obj = SneakyObject()
print("obj == None:", obj == None)   # True  (misleading)
print("obj is None:", obj is None)   # False (truth)

### ✅ Answer
```
obj == None: True
obj is None: False
```

### 🔍 Walkthrough
- `==` asks: "are these **equal**?" → runs the object's `__eq__` method, which a class can rewrite to say anything.
- `is` asks: "are these the **same object in memory**?" → can't be overridden.
- `None` is a singleton: there is only one `None` in the whole program. So `x is None` is a precise and reliable check.

### 🧠 Remember
- Use `is None` / `is not None` for `None` checks. Always.
- Use `==` for comparing values (numbers, strings, lists).
- Bonus trap: `is` on numbers/strings can give surprising results due to caching, so never use `is` to compare values.

---
# 🧠 Section 3 — Real DE Challenges

For each: **idea → solution → walkthrough**. These patterns come up again and again.

## 3.1 Group & aggregate

**Idea:** This is SQL's `GROUP BY` done by hand. Make a dict where each key is a category and each value holds a running `total` and `count`. Loop once to fill it. Then compute averages at the end.

Why not compute the average while looping? Because you don't know the final count until you've seen every row.

In [ ]:
from typing import List, Dict, Any

def aggregate_transactions(transactions: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    grouped = {}

    # Pass 1: running totals and counts
    for tx in transactions:
        cat = tx["category"]
        amt = tx["amount"]
        if cat not in grouped:
            grouped[cat] = {"total_amount": 0, "count": 0}
        grouped[cat]["total_amount"] += amt
        grouped[cat]["count"] += 1

    # Pass 2: add averages
    result = {}
    for cat, stats in grouped.items():
        result[cat] = {
            "total_amount": stats["total_amount"],
            "count": stats["count"],
            "avg_amount": round(stats["total_amount"] / stats["count"], 2),
        }
    return result

# ---- tests ----
tx_data = [
    {"category": "Electronics", "amount": 1200},
    {"category": "Groceries", "amount": 150},
    {"category": "Electronics", "amount": 800},
    {"category": "Groceries", "amount": 250},
    {"category": "Clothing", "amount": 300}
]

res = aggregate_transactions(tx_data)

assert res["Electronics"]["total_amount"] == 2000
assert res["Electronics"]["count"] == 2
assert res["Electronics"]["avg_amount"] == 1000.0

assert res["Groceries"]["total_amount"] == 400
assert res["Groceries"]["count"] == 2
assert res["Groceries"]["avg_amount"] == 200.0

assert res["Clothing"]["total_amount"] == 300
assert res["Clothing"]["count"] == 1

print("✅ 3.1 aggregate_transactions passed all tests!")
print(res)

### 🔍 Walkthrough
How `grouped` evolves as we loop:

| Row | Action | `grouped` after |
|---|---|---|
| Electronics 1200 | new key, add | `{Electronics: {total:1200, count:1}}` |
| Groceries 150 | new key, add | `+ Groceries: {150, 1}` |
| Electronics 800 | key exists, add | `Electronics: {2000, 2}` |
| Groceries 250 | key exists, add | `Groceries: {400, 2}` |
| Clothing 300 | new key, add | `+ Clothing: {300, 1}` |

Then pass 2: `Electronics avg = 2000 / 2 = 1000.0`, `Groceries avg = 400 / 2 = 200.0`, `Clothing avg = 300 / 1 = 300.0`.

### 🧠 Remember
- The `if cat not in grouped:` line creates the bucket the first time we see a category. Without it you'd get a `KeyError`.
- Shortcut once you're comfortable: `collections.defaultdict` removes that `if`. The plain-dict version above is the safest to write in an interview.

## 3.2 Enrich with a dimension lookup (in-memory LEFT JOIN)

**Idea:** Don't search the departments list for every employee (slow). Build a lookup dict **once**: `dept_id → dept_name`. Then each employee is a single instant `.get()`.

In [ ]:
from typing import List, Dict, Any

def enrich_employees(
    employees: List[Dict[str, Any]],
    departments: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    # Step 1: build the lookup table
    dept_map = {d["dept_id"]: d["dept_name"] for d in departments}

    # Step 2: enrich each employee
    result = []
    for emp in employees:
        row = dict(emp)  # copy so we don't mutate the original
        row["dept_name"] = dept_map.get(emp["dept_id"], "Unknown")
        result.append(row)
    return result

# ---- tests ----
emp_list = [
    {"emp_id": 1, "name": "Pavan", "dept_id": 10},
    {"emp_id": 2, "name": "Rohan", "dept_id": 20},
    {"emp_id": 3, "name": "Meera", "dept_id": 99}  # Not in departments table!
]

dept_list = [
    {"dept_id": 10, "dept_name": "Data Engineering"},
    {"dept_id": 20, "dept_name": "Cloud Operations"}
]

enriched = enrich_employees(emp_list, dept_list)

assert enriched[0]["dept_name"] == "Data Engineering"
assert enriched[1]["dept_name"] == "Cloud Operations"
assert enriched[2]["dept_name"] == "Unknown", "Missing dept_id should default to Unknown"

print("✅ 3.2 enrich_employees passed all tests!")
print(enriched)

### 🔍 Walkthrough
Step 1 builds:
```python
dept_map = {10: "Data Engineering", 20: "Cloud Operations"}
```
Step 2, per employee:
- Pavan → `dept_map.get(10, "Unknown")` → `"Data Engineering"`
- Rohan → `dept_map.get(20, "Unknown")` → `"Cloud Operations"`
- Meera → `dept_map.get(99, "Unknown")` → `99` isn't there → `"Unknown"` (this is the **LEFT JOIN** behavior: keep the row, fill the gap)

### 🧠 Remember
- **Why it's fast:** nested loops = N × M comparisons. Dict lookup = N + M total. Say "hash map lookup" in the interview.
- **Why `dict(emp)`:** it copies the employee so the caller's original data isn't modified behind their back. A good habit to mention.
- Using `.get(..., "Unknown")` instead of `[...]` is what makes it a left join instead of crashing on missing departments.

## 3.3 Deduplicate, keep first-seen order

**Idea:** Walk the list once. Keep a `seen` set for fast "have I seen this?" checks, and a `result` list that keeps the order.

In [ ]:
from typing import List, Any

def deduplicate_preserve_order(items: List[Any]) -> List[Any]:
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

# ---- tests ----
input_ids = [101, 105, 101, 102, 105, 103, 101]
res = deduplicate_preserve_order(input_ids)

assert res == [101, 105, 102, 103], f"Expected [101, 105, 102, 103], got {res}"
assert deduplicate_preserve_order([]) == []
assert deduplicate_preserve_order(["A", "A", "A"]) == ["A"]

print("✅ 3.3 deduplicate_preserve_order passed all tests!")

### 🔍 Walkthrough with `[101, 105, 101, 102, 105, 103, 101]`

| item | in `seen`? | action | `result` |
|---|---|---|---|
| 101 | no | add | `[101]` |
| 105 | no | add | `[101, 105]` |
| 101 | **yes** | skip | `[101, 105]` |
| 102 | no | add | `[101, 105, 102]` |
| 105 | **yes** | skip | same |
| 103 | no | add | `[101, 105, 102, 103]` |
| 101 | **yes** | skip | same |

### 🧠 Remember
- Why not `list(set(items))`? Sets don't promise to keep order.
- Why a `seen` **set** and not `if item not in result`? Checking membership in a list is slow (scans everything). A set is instant.
- One-liner alternative (works in Python 3.7+, since dicts keep insertion order):

In [ ]:
print(list(dict.fromkeys([101, 105, 101, 102, 105, 103, 101])))

Write the loop version first in an interview (it shows you understand *why*), then mention the one-liner as a bonus.

Limitation for both: items must be **hashable** (numbers, strings, tuples ✅. Lists and dicts ❌).

## 3.4 Cleaning messy records

**Idea:** For each record, *try* to convert everything. If any conversion fails or a rule is broken, skip that record and move on. `try/except` + `continue` is the star here.

In [ ]:
from typing import List, Dict, Any

def clean_sales_records(raw_records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    clean = []
    for r in raw_records:
        try:
            # price: must exist and be a valid number
            price_raw = r.get("price")
            if price_raw is None:
                continue
            price = float(str(price_raw).strip())

            # quantity: must be an int and > 0
            qty = int(r.get("quantity", 0))
            if qty <= 0:
                continue

            # id: must be an int
            record_id = int(r["id"])

            clean.append({
                "id": record_id,
                "price": price,
                "quantity": qty,
                "total_amount": round(price * qty, 2),
            })
        except (ValueError, TypeError, KeyError):
            continue   # anything malformed → skip the record
    return clean

# ---- tests ----
dirty_data = [
    {"id": "1", "price": "  150.50 ", "quantity": "2"}, # Valid -> 301.0
    {"id": "2", "price": "invalid",   "quantity": "5"}, # Invalid price -> skip
    {"id": "3", "price": "20.00",     "quantity": "0"}, # Zero qty -> skip
    {"id": "4", "price": None,        "quantity": "1"}, # None price -> skip
    {"id": "5", "price": "10.00",     "quantity": "-2"},# Negative qty -> skip
    {"id": "6", "price": "25.00",     "quantity": "4"}  # Valid -> 100.0
]

cleaned = clean_sales_records(dirty_data)
assert len(cleaned) == 2, f"Expected 2 clean records, got {len(cleaned)}"
assert cleaned[0]["id"] == 1 and cleaned[0]["total_amount"] == 301.0
assert cleaned[1]["id"] == 6 and cleaned[1]["total_amount"] == 100.0

print("✅ 3.4 clean_sales_records passed all tests!")
print(cleaned)

### 🔍 Walkthrough, record by record

| id | What happens | Result |
|---|---|---|
| 1 | `"  150.50 "` → strip → `150.5`; qty `2` > 0 → total `301.0` | ✅ kept |
| 2 | `float("invalid")` → `ValueError` → caught → skip | ❌ |
| 3 | price OK, qty `0` → `qty <= 0` → skip | ❌ |
| 4 | price is `None` → skip right away | ❌ |
| 5 | price OK, qty `-2` → `qty <= 0` → skip | ❌ |
| 6 | `25.0 × 4 = 100.0` | ✅ kept |

### 🧠 Remember
- `continue` inside `try` = "stop with this record, jump to the next one".
- `str(price_raw).strip()` first, so `float()` doesn't choke on spaces. Wrapping in `str()` also means it works if price was already a number.
- `except (ValueError, TypeError)` catches bad conversions. I added `KeyError` too, in case `id` is missing entirely.
- Don't use a bare `except:`. It hides real bugs. Catch the specific errors you expect.
- In real pipelines you'd also **log or collect the rejected rows** instead of silently dropping them. Worth saying in an interview.

## 3.5 Top earner per department (Python's `ROW_NUMBER() ... WHERE rnk = 1`)

**Idea:** Keep a dict `dept → best employee so far`. Loop once. For each employee: if their department has no leader yet, or they earn more than the current leader, they become the leader.

In [ ]:
from typing import List, Dict, Any

def top_earner_per_dept(employees: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    top_per_dept = {}
    for emp in employees:
        dept = emp["dept"]
        if dept not in top_per_dept or emp["salary"] > top_per_dept[dept]["salary"]:
            top_per_dept[dept] = emp
    return top_per_dept

# ---- tests ----
team = [
    {"name": "Pavan", "dept": "Data", "salary": 65000},
    {"name": "Sneha", "dept": "Data", "salary": 85000},
    {"name": "Amit", "dept": "QA", "salary": 45000},
    {"name": "Kiran", "dept": "QA", "salary": 52000},
    {"name": "Rahul", "dept": "DevOps", "salary": 60000}
]

top_earners = top_earner_per_dept(team)

assert top_earners["Data"]["name"] == "Sneha" and top_earners["Data"]["salary"] == 85000
assert top_earners["QA"]["name"] == "Kiran" and top_earners["QA"]["salary"] == 52000
assert top_earners["DevOps"]["name"] == "Rahul"

print("✅ 3.5 top_earner_per_dept passed all tests!")

### 🔍 Walkthrough

| Employee | Dept leader before | Decision | Leaders after |
|---|---|---|---|
| Pavan 65000 (Data) | none | first one → leader | Data: Pavan |
| Sneha 85000 (Data) | Pavan 65000 | 85000 > 65000 → replace | Data: Sneha |
| Amit 45000 (QA) | none | leader | QA: Amit |
| Kiran 52000 (QA) | Amit 45000 | 52000 > 45000 → replace | QA: Kiran |
| Rahul 60000 (DevOps) | none | leader | DevOps: Rahul |

### 🧠 Remember
- One pass → fast. No sorting needed.
- The condition `dept not in top_per_dept or ...` works because of **short-circuit `or`**: if the dept isn't there yet, Python never evaluates the second half (which would crash on a missing key).
- **Ties:** we use `>` (strictly greater), so on equal salaries the **first** one seen keeps the spot. Use `>=` if you'd want the last one. Mention this in an interview. It shows you thought about edge cases (SQL's `ROW_NUMBER` also picks arbitrarily on ties unless you add a tiebreaker).
- Alternative using `max()`:

In [ ]:
# Alternative: group first, then use max() with a key
def top_earner_per_dept_v2(employees):
    groups = {}
    for emp in employees:
        groups.setdefault(emp["dept"], []).append(emp)
    return {dept: max(members, key=lambda e: e["salary"]) for dept, members in groups.items()}

assert top_earner_per_dept_v2(team)["Data"]["name"] == "Sneha"
print("✅ v2 works too")

---
# 🎯 Cheat Sheet: Patterns to Burn Into Memory

| Pattern | Code |
|---|---|
| Sort list of dicts | `sorted(rows, key=lambda r: r["col"], reverse=True)` |
| Safe lookup | `d.get("key", default)` (but watch `None` values!) |
| Set diff / overlap | `a - b`, `a & b` |
| Parse / format date | `strptime` (string→date), `strftime` (date→string) |
| Headers + row → dict | `dict(zip(headers, row))` |
| Group & aggregate | dict accumulator, loop once, compute averages after |
| Join / enrich | build `{key: value}` lookup, then `.get(key, default)` |
| Dedupe keeping order | `seen` set + `result` list |
| Clean messy data | `try/except` + `continue`, validate rules |
| Top-1 per group | dict of "best so far", compare in a single pass |

### The 6 gotchas in one line each
1. `b = a` doesn't copy; `.copy()` is shallow; `deepcopy` for nested.
2. Default args are built once, so use `None` instead of `[]`.
3. Don't remove from a list while looping over it. Build a new list.
4. `d[key]` for required fields (fail loudly), `.get()` for optional ones.
5. Tuples can't change *which* objects they hold, but mutable objects inside can change.
6. `is None`, not `== None`.

Keep going, you've got this. 💪